# Session 1: watch a star blink
**A. Real-world problem.** Stars are far too distant to photograph their planets directly.
But if a planet's orbit is edge-on to us, the planet passes in front of its star once per
orbit and blocks a little of the starlight. Astronomers point a telescope at one star for
weeks and measure its brightness again and again. A tiny, regular dip is the sign of a planet.

**Plan.** (1) make a star's brightness record (a *light curve*) with a hidden planet,
(2) find the dips, (3) read the period from the gap between dips, (4) read the planet size
from the depth of a dip.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: how deep is the dip?
Think of the star as a bright disk and the planet as a small dark disk sliding across it.
The blocked fraction of the light is the fraction of the star's disk that is covered. For a
star of radius Rs and a planet of radius Rp, the covered fraction is the ratio of the two
disk areas:

    depth = (pi Rp^2) / (pi Rs^2) = (Rp / Rs)^2

A planet with one tenth of the star's radius covers 1/100 of the disk, so the star gets
1 percent fainter. (Session 2 checks this carefully and adds real-star effects.)

In [ ]:
def transit_depth(radius_ratio):
    """Fraction of starlight blocked by a planet with radius Rp = radius_ratio * Rs."""
    return radius_ratio ** 2

## Step 2: a star's brightness record
`light_curve` makes the brightness (1.0 = normal) every 36 minutes (0.025 days) for 12 days.
During a transit the brightness drops by `depth`. Real measurements are noisy, so we add
random noise with a fixed seed so everyone gets the same numbers.

In [ ]:
def box_dips(t, period, t0, duration, depth):
    """Brightness with a flat-bottomed dip of the given depth and duration, once per period."""
    phase = (t - t0 + period / 2) % period - period / 2     # time from the nearest transit centre
    flux = np.ones_like(t)
    flux[np.abs(phase) < duration / 2] -= depth
    return flux


def light_curve(period=3.0, t0=1.0, duration=0.15, depth=0.01, noise=0.0015, days=12.0, seed=7):
    t = np.arange(0.0, days, 0.025)
    rng = np.random.default_rng(seed)
    return t, box_dips(t, period, t0, duration, depth) + rng.normal(0.0, noise, len(t))

## Step 3: find the dips and read off the period
A point belongs to a dip if its brightness is below a *threshold* halfway between normal
(1.0) and the bottom of a dip. Neighbouring low points form one dip; we ask for at least 3
in a row so that a single noisy point is not mistaken for a planet. The period is the
typical gap between the centres of neighbouring dips.

In [ ]:
def find_dips(t, flux, threshold, min_points=3):
    """Return the centre time of each run of at least `min_points` consecutive points below threshold."""
    low = flux < threshold
    centres, start = [], None
    for i, is_low in enumerate(np.append(low, False)):
        if is_low and start is None:
            start = i
        elif not is_low and start is not None:
            if i - start >= min_points:
                centres.append(float(np.mean(t[start:i])))
            start = None
    return np.array(centres)


def period_from_dips(centres):
    """Typical gap between neighbouring dips, in days (needs at least two dips)."""
    return float(np.median(np.diff(centres))) if len(centres) >= 2 else float("nan")


def depth_from_curve(t, flux, centres, duration):
    """Average brightness drop inside the dips, measured against the average outside."""
    inside = np.zeros(len(t), dtype=bool)
    for c in centres:
        inside |= np.abs(t - c) < duration / 4
    return float(np.mean(flux[~inside & (flux > 0.9)]) - np.mean(flux[inside]))

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    t, flux = light_curve()
    centres = find_dips(t, flux, threshold=0.995)
    period = period_from_dips(centres)
    depth = depth_from_curve(t, flux, centres, 0.15)
    ratio = np.sqrt(depth)
    print(f"{len(t)} measurements, {len(centres)} dips found at {np.round(centres, 2)} days")
    print(f"period from the gaps: {period:.2f} days")
    print(f"depth: {depth:.4f} (about {depth:.1%}), radius ratio Rp/Rs = {ratio:.3f}")

    fig, ax = plt.subplots(figsize=(9, 3.8))
    ax.plot(t, flux, ".", ms=3, color="0.35", label="measured brightness")
    for c in centres:
        ax.axvline(c, color="tab:red", ls=":", lw=1)
    ax.axhline(0.995, color="tab:blue", ls="--", lw=1, label="threshold 0.995")
    ax.set_xlabel("time (days)")
    ax.set_ylabel("brightness (1 = normal)")
    ax.set_title("A light curve with a hidden planet: dips every 3 days")
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "light_curve.png", dpi=110)

    fig, ax = plt.subplots(figsize=(5.5, 4))
    ks = np.linspace(0.01, 0.2, 100)
    ax.plot(ks, 100 * transit_depth(ks), color="tab:blue")
    ax.plot([0.1], [1.0], "o", color="tab:red", label="Rp = Rs/10 blocks 1 percent")
    ax.set_xlabel("radius ratio Rp / Rs (no unit)")
    ax.set_ylabel("dip depth (percent of the starlight)")
    ax.set_title("Bigger planet, deeper dip")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "depth_vs_size.png", dpi=110)

## Where this stops
A real transit is not a flat box: the edge of a star is dimmer and the planet slides in and
out gradually. Session 2 models that and the exact duration.

## Coding tasks
1. **Easy.** In `lesson.py` change the depth to 0.004 and the period to 2.5 days. Predict the number of dips in 12 days first, then run it.
2. **Stretch.** Write `dips_per_run(period, days)` that returns how many dips you expect and compare it with `find_dips`. (Hint: it depends on where the first dip falls.)
3. **Explain.** Explain why a planet with twice the radius blocks four times as much light, not twice as much.

## Assessment
- **A. Conceptual.** Why does a transit only happen for some planets? Why do astronomers need to watch a star for longer than one orbit to be sure about a planet?
- **B. Numerical.** A planet has half the radius of its star. What fraction of the light does it block? Is that realistic for a planet?
- **C. Graph and simulation interpretation.** In the slider app the graph shows dips with period 3 days. You raise the period to 6 days but keep the planet size. What changes on the graph, and what stays the same?
- **D. Design and optimisation challenge.** Design: you can watch a star for 12 days. A planet has a period of 5 days. What is the smallest number of dips you will see, and what is the longest period for which you are guaranteed to see two dips?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("a planet with Rp/Rs = 0.1 blocks 1 percent", abs(L.transit_depth(0.1) - 0.01) < 1e-12))
    checks.append(("worked example: Rp/Rs = 0.05 blocks 0.25 percent (0.0025)", abs(L.transit_depth(0.05) - 0.0025) < 1e-12))
    checks.append(("hand calculation: a planet half the star's radius covers 25 percent", abs(L.transit_depth(0.5) - 0.25) < 1e-12))
    checks.append(("worked example: dips at days 2, 7, 12 give a 5 day period", abs(L.period_from_dips(np.array([2.0, 7.0, 12.0])) - 5.0) < 1e-12))
    checks.append(("worked example: depth 0.0025 gives Rp/Rs = 0.05", abs(np.sqrt(0.0025) - 0.05) < 1e-12))

    t = np.arange(0.0, 12.0, 0.025)
    flat = L.box_dips(t, 3.0, 1.0, 0.15, 0.01)
    checks.append(("noise-free box light curve: brightest value is exactly 1.0", abs(flat.max() - 1.0) < 1e-12))
    checks.append(("noise-free box light curve: lowest value is 1 - depth = 0.99", abs(flat.min() - 0.99) < 1e-12))
    checks.append(("noise-free curve has 4 dips", len(L.find_dips(t, flat, 0.995)) == 4))

    t, flux = L.light_curve()
    checks.append(("the light curve has 480 measurements over 12 days", len(t) == 480))
    c = L.find_dips(t, flux, 0.995)
    checks.append(("noisy curve (seed 7): 4 dips found", len(c) == 4))
    checks.append(("dip centres are near days 1, 4, 7, 10", bool(np.allclose(c, [1.0, 4.0, 7.0, 10.0], atol=0.05))))
    checks.append(("period from the gaps is 3.00 days (within 0.02)", abs(L.period_from_dips(c) - 3.0) < 0.02))
    depth = L.depth_from_curve(t, flux, c, 0.15)
    checks.append((f"measured depth is about 0.01 (got {depth:.4f})", abs(depth - 0.01) < 0.0007))
    checks.append(("radius ratio from the measured depth is about 0.1", abs(np.sqrt(depth) - 0.1) < 0.004))

    # edge cases
    rng = np.random.default_rng(3)
    quiet = 1.0 + rng.normal(0.0, 0.0015, 480)
    checks.append(("pure noise: no dips found", len(L.find_dips(np.arange(480) * 0.025, quiet, 0.995)) == 0))
    checks.append(("a single dip gives no period (nan)", np.isnan(L.period_from_dips(np.array([4.0])))))
    checks.append(("a dip of only 2 points is ignored", len(L.find_dips(np.arange(10.0), np.array([1, 1, 0.9, 0.9, 1, 1, 1, 1, 1, 1.0]), 0.95)) == 0))
    checks.append(("a dip that runs to the end of the data is still found", len(L.find_dips(np.arange(6.0), np.array([1, 1, 1, 0.9, 0.9, 0.9]), 0.95)) == 1))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
